# LocalSources / SolarFlux — teste real

Este notebook executa um teste real da entidade `SolarFlux` via:
- `SolarFluxDownloader` (download + parsing SRD/SRS),
- `SolarFluxRepository` (cache NetCDF local),
- `LocalSourcesService` (orquestração cache-first).

**Unidade canônica reportada pela entidade:** Jansky (Jy).

In [1]:
from datetime import UTC, datetime
from pathlib import Path
from importlib import reload
import sys

import xarray as xr

sys.path.insert(0, "/home/lbaros/BAROSI/PYTHONIA/radioskysim/src")

from radioskysims.domain.common import Observer, TimeRange
import radioskysims.infrastructure.external.solar_flux_downloader as solar_flux_downloader_mod
import radioskysims.infrastructure.db.solar_flux_repository as solar_flux_repository_mod
import radioskysims.application.services.local_sources_service as local_sources_service_mod
import radioskysims.infrastructure.external.local_source_track_skyfield_provider as local_track_provider_mod

# Garante uso da versão mais recente do código durante iteração no notebook
solar_flux_downloader_mod = reload(solar_flux_downloader_mod)
solar_flux_repository_mod = reload(solar_flux_repository_mod)
local_sources_service_mod = reload(local_sources_service_mod)
local_track_provider_mod = reload(local_track_provider_mod)

SolarFluxDownloader = solar_flux_downloader_mod.SolarFluxDownloader
SolarFluxRepository = solar_flux_repository_mod.SolarFluxRepository
LocalSourcesService = local_sources_service_mod.LocalSourcesService
SkyfieldLocalSourceTrackProvider = (
    local_track_provider_mod.SkyfieldLocalSourceTrackProvider
)

In [2]:
cache_dir = Path("data/aux/signals/solar_flux")
repo = SolarFluxRepository(engine=None, cache_dir=cache_dir)
downloader = SolarFluxDownloader(timeout_sec=30)
track_provider = SkyfieldLocalSourceTrackProvider()
service = LocalSourcesService(
    solar_flux_downloader=downloader,
    solar_flux_repository=repo,
    local_source_track_provider=track_provider,
    default_window_hours=2,
)

local_sources = service.build_local_sources()
local_sources

LocalSources(sources=[LocalSource(name='Sun', kind='sun', has_track=True, has_signal=True)])

In [3]:
from datetime import timedelta

window_hours = 2
center_utc = datetime.now(UTC)

# Tenta centro atual e recua em passos de 6h até 72h para trás
solar_flux = None
center_used_utc = None
last_error = None

for step_h in range(0, 73, 6):
    candidate_center = center_utc - timedelta(hours=step_h)
    try:
        solar_flux = service.get_solar_flux(
            center_utc=candidate_center,
            window_hours=window_hours,
        )
        center_used_utc = candidate_center
        break
    except RuntimeError as exc:
        last_error = exc

if solar_flux is None:
    raise RuntimeError(
        f"Falha ao obter SolarFlux nas últimas 72h. Último erro: {last_error}"
    )

print("Centro solicitado:", center_utc)
print("Centro usado:", center_used_utc)
print("Intervalo retornado:", solar_flux.start_utc, "->", solar_flux.end_utc)
print("Origem:", solar_flux.source_url)
print("Variáveis:", list(solar_flux.dataset.data_vars))
print("Dimensões:", dict(solar_flux.dataset.sizes))
solar_flux.dataset

Centro solicitado: 2026-07-27 15:05:05.592274+00:00
Centro usado: 2026-07-27 03:05:05.592274+00:00
Intervalo retornado: 2026-07-27 05:05:05.592274 -> 2026-07-27 07:05:05.592274
Origem: https://downloads.sws.bom.gov.au/wdc/wdc_solradio/data/learmonth/SRD/2026/L260726.SRD
Variáveis: ['flux_jy']
Dimensões: {'time': 7192, 'frequency_mhz': 8}


<xarray.Dataset> Size: 518kB
Dimensions:        (time: 7192, frequency_mhz: 8)
Coordinates:
  * time           (time) datetime64[ns] 58kB 2026-07-27T05:05:06 ... 2026-07...
  * frequency_mhz  (frequency_mhz) float64 64B 245.0 410.0 ... 8.8e+03 1.54e+04
Data variables:
    flux_jy        (time, frequency_mhz) float64 460kB dask.array<chunksize=(7192, 8), meta=np.ndarray>
Attributes:
    units:    Jy
    source:   SRD

In [4]:
import numpy as np

freqs = np.array(solar_flux.dataset["frequency_mhz"].values, dtype=float)
freqs_sorted = np.sort(freqs)
print("Frequências disponíveis (MHz):", freqs_sorted.tolist())

target_freqs = [245.0, 1415.0]
for target in target_freqs:
    nearest = float(freqs_sorted[np.argmin(np.abs(freqs_sorted - target))])
    flux_jy = solar_flux.evaluate(center_used_utc.replace(tzinfo=None), nearest)
    print(
        f"Fluxo interpolado ~{target:.0f} MHz (usando {nearest:.0f} MHz): {flux_jy:.3f} Jy"
    )

Frequências disponíveis (MHz): [245.0, 410.0, 610.0, 1415.0, 2695.0, 4995.0, 8800.0, 15400.0]
Fluxo interpolado ~245 MHz (usando 245 MHz): 250000.000 Jy
Fluxo interpolado ~1415 MHz (usando 1415 MHz): 4077.260 Jy


In [5]:
# Visualização rápida (tempo x frequência)
flux_da = solar_flux.dataset["flux_jy"]
flux_da.compute()

<xarray.DataArray 'flux_jy' (time: 7192, frequency_mhz: 8)> Size: 460kB
array([[2.40e+05, 4.40e+05, 7.30e+05, ..., 1.00e+04, 0.00e+00, 4.65e+07],
       [2.40e+05, 4.40e+05, 7.30e+05, ..., 1.00e+04, 0.00e+00, 4.65e+07],
       [2.40e+05, 4.40e+05, 7.30e+05, ..., 1.00e+04, 0.00e+00, 4.65e+07],
       ...,
       [2.80e+05, 4.70e+05, 7.20e+05, ..., 0.00e+00, 0.00e+00, 4.65e+07],
       [2.80e+05, 4.80e+05, 7.10e+05, ..., 0.00e+00, 0.00e+00, 4.65e+07],
       [2.80e+05, 4.70e+05, 7.10e+05, ...,      nan, 0.00e+00, 4.65e+07]],
      shape=(7192, 8))
Coordinates:
  * time           (time) datetime64[ns] 58kB 2026-07-27T05:05:06 ... 2026-07...
  * frequency_mhz  (frequency_mhz) float64 64B 245.0 410.0 ... 8.8e+03 1.54e+04

In [6]:
# Track do Sol para o observador (exemplo: Uirapuru/BINGO ~ PB)
from datetime import timedelta

start_track = center_used_utc.replace(tzinfo=None) - timedelta(minutes=30)
end_track = center_used_utc.replace(tzinfo=None) + timedelta(minutes=30)

tr_track = TimeRange.from_iso(
    start=start_track.strftime("%Y-%m-%d %H:%M:%S"),
    end=end_track.strftime("%Y-%m-%d %H:%M:%S"),
    resolution=60.0,
)

observer = Observer(
    lat=-7.25,
    lon=-35.86,
    height=550.0,
    timeframe=tr_track,
)

sun_track = service.get_local_source_track(
    source_name="Sun",
    observer=observer,
    time_range=tr_track,
)

print("Variáveis track:", list(sun_track.dataset.data_vars))
print("Dimensões track:", dict(sun_track.dataset.sizes))
sun_track.dataset

Variáveis track: ['theta_rad', 'az_rad', 'el_rad', 'range_m']
Dimensões track: {'time': 61}


<xarray.Dataset> Size: 2kB
Dimensions:    (time: 61)
Coordinates:
  * time       (time) datetime64[ns] 488B 2026-07-27T02:33:55.815981821 ... 2...
Data variables:
    theta_rad  (time) float64 488B 2.932 2.932 2.931 2.931 ... 2.806 2.803 2.8
    az_rad     (time) float64 488B 0.07788 0.09757 0.1172 ... 0.8796 0.8868
    el_rad     (time) float64 488B -1.361 -1.361 -1.36 ... -1.235 -1.232 -1.229
    range_m    (time) float64 488B 1.519e+11 1.519e+11 ... 1.519e+11 1.519e+11
Attributes:
    provider:  skyfield
    source:    Sun